# Notebook 02 — Multi-Stage Custom Augmentation Transforms

**Session:** Image Augmentation  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement a composable stochastic image augmentation pipeline from first principles (affine scale/translate/rotate, Gaussian/average/median/motion blur, additive Gaussian noise, brightness shift, and linear contrast) wrapped as a callable PyTorch transform.

**Learning objectives:**
- Build `MixAugment` and `ConvertToTensor` callable transform classes.
- Implement geometric affine warps and spatial blur kernels directly with OpenCV and NumPy.
- Train a ResNet-18 classifier with K-Fold cross-validation using custom transforms.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from torchvision import transforms
from torchvision.models import resnet18

## 1  Training, Validation, and Plotting Utilities

In [ ]:
def get_accuracy(model: nn.Module, data_loader: DataLoader, device: torch.device) -> float:
    """Compute classification accuracy (%) over an entire DataLoader."""
    correct = 0
    total = 0

    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100.0 * (correct / total) if total > 0 else 0.0


def plot_losses(
    train_acc: list[float],
    valid_acc: list[float],
    train_loss: list[float],
    valid_loss: list[float],
) -> None:
    """Plot training and validation accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(train_acc, color="tab:blue", label="Train Accuracy")
    ax1.plot(valid_acc, color="tab:red", label="Validation Accuracy")
    ax1.set(title="Accuracy over Epochs", xlabel="Epoch", ylabel="Accuracy (%)")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(train_loss, color="tab:blue", label="Train Loss")
    ax2.plot(valid_loss, color="tab:red", label="Validation Loss")
    ax2.set(title="Loss over Epochs", xlabel="Epoch", ylabel="Loss")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    fig.tight_layout()
    plt.show()


def validate(
    valid_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[nn.Module, float]:
    """Run one validation epoch without gradient tracking."""
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

    epoch_loss = running_loss / max(len(valid_loader), 1)
    return model, epoch_loss


def train_one_epoch(
    train_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[nn.Module, torch.optim.Optimizer, float]:
    """Run one supervised training epoch."""
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model, optimizer, running_loss / max(len(train_loader), 1)


def training_loop(
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    epochs: int,
    device: torch.device,
    fold_idx: int,
    checkpoint_dir: Path = Path("outputs/checkpoints"),
) -> tuple[nn.Module, torch.optim.Optimizer, tuple[list[float], list[float]]]:
    """Execute the full training loop and plot learning curves."""
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    train_losses: list[float] = []
    valid_losses: list[float] = []
    list_train_acc: list[float] = []
    list_val_acc: list[float] = []

    for epoch in range(epochs):
        model, optimizer, train_loss = train_one_epoch(
            train_loader, model, criterion, optimizer, device
        )
        model, valid_loss = validate(valid_loader, model, criterion, device)
        train_acc = get_accuracy(model, train_loader, device=device)
        valid_acc = get_accuracy(model, valid_loader, device=device)

        print(
            f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | Valid loss: {valid_loss:.4f} "
            f"| Train acc: {train_acc:.2f}% | Valid acc: {valid_acc:.2f}%"
        )
        list_train_acc.append(train_acc)
        list_val_acc.append(valid_acc)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        torch.save(
            model.state_dict(),
            checkpoint_dir / f"custom_aug_fold_{fold_idx + 1}_epoch_{epoch}.pth",
        )

    plot_losses(list_train_acc, list_val_acc, train_losses, valid_losses)
    return model, optimizer, (train_losses, valid_losses)

## 2  Custom Stochastic Augmentation Pipeline

In [ ]:
class SampleDataset(Dataset):
    """Dataset returning raw (H, W, C) uint8 RGB image and integer label tuples."""

    def __init__(self, root_dir: str | Path, transform=None) -> None:
        self.root_dir = Path(root_dir)
        self.list_images_path: list[Path] = []
        self.list_labels: list[str] = []
        self.class_to_idx = {"dogs": 0, "cats": 1}
        self.transform = transform

        for sub_dir in sorted(self.root_dir.iterdir()):
            if not sub_dir.is_dir() or sub_dir.name not in self.class_to_idx:
                continue
            for image_path in sorted(sub_dir.iterdir()):
                if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.list_images_path.append(image_path)
                    self.list_labels.append(sub_dir.name)

    def __len__(self) -> int:
        return len(self.list_images_path)

    def __getitem__(self, idx: int):
        image = cv2.imread(str(self.list_images_path[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        label = np.array(self.class_to_idx[self.list_labels[idx]], dtype=np.int64)

        sample = (image, label)
        if self.transform is not None:
            sample = self.transform(sample)
        return sample


class ConvertToTensor:
    """Convert a (H x W x C uint8 image, label) tuple to normalized PyTorch tensors."""

    def __call__(self, sample: tuple[np.ndarray, np.ndarray]) -> tuple[torch.Tensor, torch.Tensor]:
        image, label = sample
        tensor_img = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        tensor_lbl = torch.from_numpy(label).long()
        return tensor_img, tensor_lbl


class MixAugment:
    """Stochastic OpenCV/NumPy pipeline mirroring affine, blur, noise, and contrast transforms."""

    def __init__(self, prob: float = 0.5, seed: int | None = None) -> None:
        self.prob = prob
        self.rng = np.random.default_rng(seed)

    def _random_affine(self, image: np.ndarray) -> np.ndarray:
        h, w = image.shape[:2]
        angle = float(self.rng.uniform(-25.0, 25.0))
        scale = float(self.rng.uniform(0.8, 1.2))
        tx = float(self.rng.uniform(-0.2, 0.2)) * w
        ty = float(self.rng.uniform(-0.2, 0.2)) * h
        matrix = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, scale)
        matrix[0, 2] += tx
        matrix[1, 2] += ty
        return cv2.warpAffine(image, matrix, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    def _random_blur(self, image: np.ndarray) -> np.ndarray:
        choice = int(self.rng.integers(0, 4))
        k = int(self.rng.choice([3, 5, 7]))
        if choice == 0:
            return cv2.GaussianBlur(image, (k, k), 0)
        if choice == 1:
            return cv2.blur(image, (k, k))
        if choice == 2:
            return cv2.medianBlur(image, k)
        kernel = np.zeros((k, k), dtype=np.float32)
        kernel[k // 2, :] = 1.0 / k
        return cv2.filter2D(image, -1, kernel)

    def _random_photometric(self, image: np.ndarray) -> np.ndarray:
        out = image.astype(np.float32)
        if self.rng.random() < 0.5:
            noise = self.rng.normal(0.0, self.rng.uniform(0.0, 15.0), size=out.shape)
            out = out + noise
        if self.rng.random() < 0.5:
            beta = float(self.rng.uniform(-35.0, 35.0))
            out = out + beta
        if self.rng.random() < 0.5:
            alpha = float(self.rng.uniform(0.6, 1.6))
            out = alpha * (out - 127.5) + 127.5
        return np.clip(out, 0, 255).astype(np.uint8)

    def __call__(self, sample: tuple[np.ndarray, np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
        image, label = sample
        aug = image.copy()
        if self.rng.random() < self.prob:
            aug = self._random_affine(aug)
        if self.rng.random() < self.prob:
            aug = self._random_blur(aug)
        if self.rng.random() < self.prob:
            aug = self._random_photometric(aug)
        return aug, label

## 3  Training with Custom Augmentations

In [ ]:
class Net(nn.Module):
    """Lightweight ResNet-18 classifier with a 3x3 input stem for 2-class tasks."""

    def __init__(self, num_classes: int = 2) -> None:
        super().__init__()
        self.model = resnet18(weights=None, num_classes=num_classes)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


DATA_DIR = Path("../data/sample_dataset")
train_transform = transforms.Compose([MixAugment(prob=0.5, seed=42), ConvertToTensor()])
val_transform = transforms.Compose([ConvertToTensor()])

transformed_train_data = SampleDataset(DATA_DIR / "train", transform=train_transform)
transformed_test_data = SampleDataset(DATA_DIR / "test", transform=val_transform)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splits = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(
    splits.split(np.arange(len(transformed_train_data)))
):
    print(f"\n=== Fold {fold_idx + 1} / 5 ===")
    train_loader = DataLoader(
        transformed_train_data, batch_size=8, sampler=SubsetRandomSampler(train_idx.tolist())
    )
    val_loader = DataLoader(
        transformed_train_data, batch_size=8, sampler=SubsetRandomSampler(val_idx.tolist())
    )
    model = Net().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    model, optimizer, _ = training_loop(
        model,
        criterion,
        optimizer,
        train_loader,
        val_loader,
        epochs=3,
        device=device,
        fold_idx=fold_idx,
    )